# Session 10: Using Ragas to Evaluate a RAG Application built with LangChain and LangGraph

In the following notebook, we'll be looking at how [Ragas](https://github.com/explodinggradients/ragas) can be helpful in a number of ways when looking to evaluate your RAG applications!

While this example is rooted in LangChain/LangGraph - Ragas is framework agnostic (you don't even need to be using a framework!).

## 🤝 Breakout Room #1
  - Task 1: Installing Required Libraries
  - Task 2: Set Environment Variables
  - Task 3: Synthetic Dataset Generation for Evaluation using Ragas
  - Task 4: Construct our RAG application
  - Task 5: Evaluating our Application with Ragas
  - Task 6: Making Adjustments and Re-Evaluating
  - ***Activity #1: Implement a Different Reranking Strategy***


## Task 1: Installing Required Libraries

If you have not already done so, install the required libraries using the uv package manager:
``` bash

uv sync

```


## Task 2: Set Environment Variables:

We'll also need to provide our API keys.
> NOTE: In addition to OpenAI's models, this notebook will be using Cohere's Reranker - please be sure to [sign-up for an API key!](https://docs.cohere.com/reference/about)

You have two options for supplying your API keys in this session:
- Use environment variables (see Prerequisite #2 in the README.md)
- Provide them via a prompt when the notebook runs

The following code will load all of the environment variables in your `.env`. Then, it checks for the two API keys we need. If they are not there, it will prompt you to provide them.

First, OpenAI's for our LLM/embedding model combination!

Second, Cohere's for our reranking


In [40]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Please enter your OpenAI API key!")

if not os.environ.get("COHERE_API_KEY"):
    os.environ["COHERE_API_KEY"] = getpass("Please enter your Cohere API key!")

## Task 3: Synthetic Dataset Generation for Evaluation using Ragas

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data using the Health & Wellness Guide - a comprehensive resource covering exercise, nutrition, sleep, and stress management.

Next, let's load our data into a familiar LangChain format using the `TextLoader`.

In [2]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [3]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

/Users/nikos/n/rvm/AIE9/10_Evaluating_RAG_With_Ragas/.venv/lib/python3.13/site-packages/pysbd/segmenter.py:66: SyntaxWarning: invalid escape sequence '\s'
  for match in re.finditer('{0}\s*'.format(re.escape(sent)), self.original_text):
/Users/nikos/n/rvm/AIE9/10_Evaluating_RAG_With_Ragas/.venv/lib/python3.13/site-packages/pysbd/lang/arabic.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)
/Users/nikos/n/rvm/AIE9/10_Evaluating_RAG_With_Ragas/.venv/lib/python3.13/site-packages/pysbd/lang/persian.py:29: SyntaxWarning: invalid escape sequence '\.'
  txt = re.sub('(?<={0})\.'.format(am), '∯', txt)


In [4]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/1 [00:00<?, ?it/s]

Applying SummaryExtractor:   0%|          | 0/1 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/4 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/9 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/1 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/2 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

In [5]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,How can shoulder shrugs help alleviate neck an...,[The Personal Wellness Guide A Comprehensive R...,Shoulder shrugs can provide relief from neck a...,single_hop_specifc_query_synthesizer
1,What is the Cat-Cow Stretch and how can it hel...,[The Personal Wellness Guide A Comprehensive R...,The Cat-Cow Stretch is a recommended exercise ...,single_hop_specifc_query_synthesizer
2,Wut is CBT-I and howw can it help with insomny...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,"CBT-I, or Cognitive Behavioral Therapy for Ins...",single_hop_specifc_query_synthesizer
3,what non-REM mean for sleep and how it help me...,[PART 3: SLEEP AND RECOVERY Chapter 7: The Sci...,"Sleep happens in cycles of about 90 minutes, s...",single_hop_specifc_query_synthesizer
4,As a wellness-oriented professional seeking to...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,PART 7 recommends several strategies for achie...,single_hop_specifc_query_synthesizer
5,What practical strategies are outlined in PART...,[PART 5: BUILDING HEALTHY HABITS Chapter 13: T...,PART 7 provides several strategies for wellnes...,single_hop_specifc_query_synthesizer
6,how chapter 7 and chapter 19 help busy pro get...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,chapter 7 say sleep is crucial for health and ...,multi_hop_specific_query_synthesizer
7,"According to Chapter 8 and Chapter 15, what ar...",[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,Chapter 8 emphasizes the importance of sleep h...,multi_hop_specific_query_synthesizer
8,How does the information in Chapter 7 about th...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,Chapter 7 explains that sleep is crucial for p...,multi_hop_specific_query_synthesizer
9,How can the strategies for managing insomnia d...,[<1-hop>\n\nPART 5: BUILDING HEALTHY HABITS Ch...,The strategies for managing insomnia in Chapte...,multi_hop_specific_query_synthesizer


## Task 4: Construct our RAG application

Now we'll construct our LangChain RAG, which we will be evaluating using the above created test data!

### R - Retrieval

Let's start with building our retrieval pipeline, which will involve loading the same data we used to create our synthetic test set above.

> NOTE: We need to use the same data - as our test set is specifically designed for this data.

In [6]:
loader = TextLoader("data/HealthWellnessGuide.txt")
docs = loader.load()

Now that we have our data loaded, let's split it into chunks!

In [7]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=50, chunk_overlap=0)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

447

### ❓ Question #1:

What is the purpose of the `chunk_overlap` parameter in the `RecursiveCharacterTextSplitter`?

##### Answer:

The splitter will first split the doc into chunks respecting the sentence/paragraph boundaries until it makes the chunk fit the expected size. Then it will append `chunk_overlap` characters to each chunk from the previous one. The splitter does not respect the sentence boundaries during that append so it's possible to have a chunk contain things mid-sentence. This probably doesn't matter for accuracy when we embed the chunks because we will still capture some similarity between the adjacent chunks which will still bring them "closer" to each other as we build our knowledge graph.

Next up, we'll need to provide an embedding model that we can use to construct our vector store.

In [8]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

Now we can build our in memory QDrant vector store.

In [9]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data",
    embedding=embeddings,
)

We can now add our documents to our vector store.

In [10]:
_ = vector_store.add_documents(documents=split_documents)

Let's define our retriever.

In [11]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

Now we can produce a node for retrieval!

In [12]:
def retrieve(state):
  retrieved_docs = retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

### A - Augmented

Let's create a simple RAG prompt!

In [13]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
You are a helpful assistant who answers questions based on provided context. You must only use the provided context, and cannot use your own knowledge.

### Question
{question}

### Context
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

### G - Generation

We'll also need an LLM to generate responses - we'll use `gpt-4o-nano` to avoid using the same model as our judge model.

In [14]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-nano")

Then we can create a `generate` node!

In [15]:
def generate(state):
  docs_content = "\n\n".join(doc.page_content for doc in state["context"])
  messages = rag_prompt.format_messages(question=state["question"], context=docs_content)
  response = llm.invoke(messages)
  return {"response" : response.content}

### Building RAG Graph with LangGraph

Let's create some state for our LangGraph RAG graph!

In [16]:
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from langchain_core.documents import Document

class State(TypedDict):
  question: str
  context: List[Document]
  response: str

Now we can build our simple graph!

> NOTE: We're using `add_sequence` since we will always move from retrieval to generation. This is essentially building a chain in LangGraph.

In [17]:
graph_builder = StateGraph(State).add_sequence([retrieve, generate])
graph_builder.add_edge(START, "retrieve")
graph = graph_builder.compile()

Let's do a test to make sure it's doing what we'd expect.

In [18]:
response = graph.invoke({"question" : "What exercises help with lower back pain?"})

In [19]:
response["response"]

'The provided context does not specify which exercises help with lower back pain.'

## Task 5: Evaluating our Application with Ragas

Now we can finally do our evaluation!

We'll start by running the queries we generated usign SDG above through our application to get context and responses.

In [20]:
for test_row in dataset:
  response = graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

In [21]:
dataset.samples[0].eval_sample.response

'Shoulder shrugs can help alleviate neck and shoulder tension for busy professionals by raising the shoulders toward the ears, which can help release built-up tension in that area. These exercises promote relaxation of the neck and shoulder muscles, reducing discomfort and stiffness caused by prolonged periods of sitting or stress.'

Then we can convert that table into a `EvaluationDataset` which will make the process of evaluation smoother.

In [22]:
from ragas import EvaluationDataset

evaluation_dataset = EvaluationDataset.from_pandas(dataset.to_pandas())

We'll need to select a judge model - in this case we're using the same model that was used to generate our Synthetic Data.

In [23]:
from ragas import evaluate
from ragas.llms import LangchainLLMWrapper

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

Next up - we simply evaluate on our desired metrics!

In [24]:
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness, ResponseRelevancy, ContextEntityRecall, NoiseSensitivity
from ragas import evaluate, RunConfig

custom_run_config = RunConfig(timeout=360)

baseline_result = evaluate(
    dataset=evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
baseline_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.1667, 'faithfulness': 0.3631, 'factual_correctness': 0.5027, 'answer_relevancy': 0.4314, 'context_entity_recall': 0.2166, 'noise_sensitivity_relevant': 0.0364}

## Task 6: Making Adjustments and Re-Evaluating

Now that we've got our baseline - let's make a change and see how the model improves or doesn't improve!




We'll first set our retriever to return more documents, which will allow us to take advantage of the reranking.

In [25]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=30)
split_documents = text_splitter.split_documents(docs)
len(split_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new_chunks",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new_chunks",
    embedding=embeddings,
)

_ = vector_store.add_documents(documents=split_documents)

adjusted_example_retriever = vector_store.as_retriever(search_kwargs={"k": 20})

Reranking, or contextual compression, is a technique that uses a reranker to compress the retrieved documents into a smaller set of documents.

This is essentially a slower, more accurate form of semantic similarity that we use on a smaller subset of our documents.

In [42]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

def retrieve_adjusted(state):
  compressor = CohereRerank(model="rerank-v3.5")
  compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=adjusted_example_retriever, search_kwargs={"k": 5}
  )
  retrieved_docs = compression_retriever.invoke(state["question"])
  return {"context" : retrieved_docs}

We can simply rebuild our graph with the new retriever!

In [43]:
class AdjustedState(TypedDict):
  question: str
  context: List[Document]
  response: str

adjusted_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_adjusted, generate])
adjusted_graph_builder.add_edge(START, "retrieve_adjusted")
adjusted_graph = adjusted_graph_builder.compile()

In [44]:
response = adjusted_graph.invoke({"question" : "How can I improve my sleep quality?"})
response["response"]

'To improve your sleep quality, consider practicing good sleep hygiene habits such as maintaining a consistent sleep schedule, creating a relaxing bedtime routine (like reading or gentle stretching), and ensuring your bedroom is cool, dark, and quiet. Limit screen exposure 1-2 hours before bed and avoid caffeine after 2 PM. Regular exercise, but not too close to bedtime, can also help. Additionally, using natural remedies like herbal teas (chamomile or valerian root), relaxation techniques, or magnesium supplements may support better sleep. Following these practices can promote more consistent and restful sleep.'

In [45]:
import time
import copy

rerank_dataset = copy.deepcopy(dataset)

for test_row in rerank_dataset:
  response = adjusted_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.

In [34]:
rerank_dataset.samples[0].eval_sample.response

'To properly perform the Bird Dog exercise, follow these steps:\n\n1. Start on your hands and knees on a comfortable, flat surface.\n2. Engage your core muscles to maintain stability throughout the movement.\n3. Simultaneously extend your opposite arm and leg — for example, extend your right arm forward and your left leg backward.\n4. Keep your back flat and avoid arching or sagging.\n5. Hold this extended position for about 5 seconds, focusing on maintaining good balance and core engagement.\n6. Return your hand and knee to the starting position.\n7. Repeat the movement on the opposite side, extending your left arm and right leg.\n8. Do 10 repetitions per side, ensuring controlled and steady movements.\n\n**Benefits for Lower Back Pain Relief:**\nThe Bird Dog exercise helps strengthen the core muscles, including the lower back and abdominals, which are essential for supporting your spine. By improving core stability, it reduces unnecessary stress on the lower back, alleviating discomf

In [35]:
rerank_evaluation_dataset = EvaluationDataset.from_pandas(rerank_dataset.to_pandas())

In [36]:
rerank_result = evaluate(
    dataset=rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
rerank_result

Evaluating:   0%|          | 0/54 [00:00<?, ?it/s]

{'context_recall': 0.9630, 'faithfulness': 0.7518, 'factual_correctness': 0.7267, 'answer_relevancy': 0.9521, 'context_entity_recall': 0.4537, 'noise_sensitivity_relevant': 0.0171}

### ❓ Question #2:

Which system performed better, on what metrics, and why?

##### Answer:

The Reranked system did better:
- `context_recall` had the highest jump +0.8 which means we got the right context pulled out of the pool. That makes sense since we increased the chunk size by 10x and also had some overlap. But I think the reranking must have helped substiantially bring the most relevant docs up to the top.
- `faithfulness` also went up substantially which is interesting because we didn't change the LLM prompt but it was able to generate its answer using the facts better. Maybe because we had more accurate facts retrieved it was able to form cleaner answers that were "inspired" by the facts instead of hallucinations.
- `answer_relevancy` had a big jump +0.5 and since we didn't modify our model prompt we can only say that fetching more accurate chunks the answer quality itself increased possibly because the first system did not have much to go with to begin with.
- `context_entity_recall` also increased (doubled) but it is still less then 50% which means that we are still missing a deep understanding of the concepts in the doc. Maybe having a bigger embedding model could help there?
- `factual_correctness` also went up by +0.2 which is not a big jump and that probably points to the fact that our LLM instructions are not great.

### ❓ Question #3:

What are the benefits and limitations of using synthetic data generation for RAG evaluation? Consider both the practical advantages and potential pitfalls.

##### Answer:

###### Benefits:
- No human in the loop so it's cheap and fast
- We can leverage existing documents we have and we don't need to produce any more "knowledge"
- We leverage the LLM's apparent ability to keep a "clear head" when switching roles from generator to evaluator. If it were not able to separate these skills then we'd have tainted "labels" or "overfitting" issues.

###### Pitfalls:
- Since we don't have real user data we will only reach a local optimum with this approach. There is so far the LLM can avoid "overfitting".
- Real user usage might be completely different from the generated questions (language, tone, errors, multi-step conversations etc.). SGD cannot capture that.

### ❓ Question #4:

If you were building a production wellness assistant, which Ragas metrics would be most important to optimize for and why? Consider the healthcare/wellness domain specifically.

##### Answer:

- `factual_correctness`: by far this is the most important metric. Wellness responses need to be of the highest accuracy since we may aversely affect someone's health.
- `answer_relevancy`: even if we are correct, if our answer includes irrelevant information, the user might still think it's relevant (e.g., Trump saying that drinking detergent prevents COVID). Imagine we had the correct answers and then we had an extra tidbit like this on our long answer. There are people who would take it seriously even if it sounds ridiculous.

## Activity #1: Implement a Different Reranking Strategy

In this activity, you'll experiment with different reranking parameters or strategies to see how they affect the evaluation metrics.

**Requirements:**
1. Modify the `retrieve_adjusted` function to use different parameters (e.g., change `k` values, try different top_n for reranking)
2. Or implement a different retrieval enhancement strategy (e.g., hybrid search, query expansion)
3. Run the evaluation and compare results with the baseline and reranking results above
4. Document your findings in the markdown cell below

In [47]:
### YOUR CODE HERE ###

# Implement your custom retrieval strategy here
# Example: modify retrieve_adjusted with different parameters
# We bump up the chunk size and overlap to get more context
custom_text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
custom_split_documents = custom_text_splitter.split_documents(docs)
len(custom_split_documents)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

client = QdrantClient(":memory:")

client.create_collection(
    collection_name="use_case_data_new_chunks_custom",
    vectors_config=VectorParams(size=1536, distance=Distance.COSINE),
)

custom_vector_store = QdrantVectorStore(
    client=client,
    collection_name="use_case_data_new_chunks_custom",
    embedding=embeddings,
)

_ = custom_vector_store.add_documents(documents=custom_split_documents)

custom_example_retriever = custom_vector_store.as_retriever(search_kwargs={"k": 20})

def retrieve_custom(state):
    compressor = CohereRerank(model="rerank-v3.5")
    custom_compression_retriever = ContextualCompressionRetriever(
        base_compressor=compressor, base_retriever=custom_example_retriever, search_kwargs={"k": 5}
    )
    retrieved_docs = custom_compression_retriever.invoke(state["question"])
    return {"context" : retrieved_docs}

custom_graph_builder = StateGraph(AdjustedState).add_sequence([retrieve_custom, generate])
custom_graph_builder.add_edge(START, "retrieve_custom")
custom_graph = custom_graph_builder.compile()

# Let's run the tests
custom_rerank_dataset = copy.deepcopy(dataset)

for test_row in custom_rerank_dataset:
  response = custom_graph.invoke({"question" : test_row.eval_sample.user_input})
  test_row.eval_sample.response = response["response"]
  test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]
  time.sleep(2) # To try to avoid rate limiting.

custom_rerank_evaluation_dataset = EvaluationDataset.from_pandas(custom_rerank_dataset.to_pandas())

custom_rerank_result = evaluate(
    dataset=custom_rerank_evaluation_dataset,
    metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness(), ResponseRelevancy(), ContextEntityRecall(), NoiseSensitivity()],
    llm=evaluator_llm,
    run_config=custom_run_config
)
custom_rerank_result

Evaluating:   0%|          | 0/66 [00:00<?, ?it/s]

{'context_recall': 0.8303, 'faithfulness': 0.6541, 'factual_correctness': 0.6291, 'answer_relevancy': 0.9523, 'context_entity_recall': 0.5020, 'noise_sensitivity_relevant': 0.1266}

### Activity #1 Findings:

*Document your findings here: What strategy did you try? How did it compare to the baseline and reranking results?*

I tried a single change in the chunking 500->1000 (2x) and the overlap 30->100 (3x). I thought it would improve the context metrics compared to the reranked version but it seems like it didn't if we look at the table below:

| Metric | Baseline | Reranked | Custom |
|--------|----------|----------|--------|
| **context_recall** | 0.1667 | 0.9630 | 0.8303 |
| **faithfulness** | 0.3631 | 0.7518 | 0.6541 |
| **factual_correctness** | 0.5027 | 0.7267 | 0.6291 |
| **answer_relevancy** | 0.4314 | 0.9521 | 0.9523 |
| **context_entity_recall** | 0.2166 | 0.4537 | 0.5020 |
| **noise_sensitivity_relevant** | 0.0364 | 0.0171 | 0.1266 |

All metrics look lower except for the improvement in `context_entity_recall` by 10% compared to the reranked version. Which means we were able to pull more conceptual entities because of the chunk size. But it seems that making the chunk too big we lose accuracy and I'm guessing that's because our wellness guide is not a lot of data so we are over-grouping knowledge.

Note: the approach is still way better than the Baseline in all aspects.
